<a href="https://colab.research.google.com/github/nawroz-m/ML_learning/blob/BigImgSingnal/boston_exercise_partial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import matplotlib.pyplot as plt
from IPython.core.debugger import Tracer
from google.colab import drive

In [2]:
# load the dataset
drive.flush_and_unmount()
drive.mount('/content/drive')

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/Big Img Sig/BostonHousingDataset"

# Definition of parameters

In [4]:
device = 'cuda'
learning_rate = 0.00001
batch_size = 50
experiment_name = 'prova1'

# Data loading

In [ ]:
class Dataset(torch.utils.data.Dataset):

    def __init__(self, csv):
        # read the csv file
        self.df = pd.read_csv(csv, sep='\s+') # manually superate the spaces
        # self.df = self.df.dropna(axis=0)
        # save cols
        self.input_cols = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT']
        self.output_cols = ['MEDV']



    def __len__(self):
        # TODO: here i will return the number of samples in the dataset
        return len(self.df)


    def __getitem__(self, idx):
        # read row, split in input and output and convert in tensors
        cur_sample=self.df.iloc[idx]
        # split output and input
        cur_sample_x = cur_sample[self.input_cols]
        # ground througth
        cur_sample_y = cur_sample[self.output_cols]

        # Conver tin torch
        cur_sample_x = torch.tensor(cur_sample_x)
        cur_sample_y = torch.tensor(cur_sample_y)
        # Conver in float

        cur_sample_x = cur_sample_x.float()
        cur_sample_y = cur_sample_y.float()

        return cur_sample_x, cur_sample_y



In [ ]:
# try to use the dataset
ds = Dataset(f'{dataset_path}/train.csv')
print(ds.__len__())
# get first item
xx,yy = ds.__getitem__(0)
# print shapes
print(xx.shape)
print(yy.shape)

In [7]:
# create train and validation datasets
train_ds = Dataset(f'{dataset_path}/train.csv')
val_ds =  Dataset(f'{dataset_path}/val.csv')

In [ ]:
# create train dataloader
train_dl = torch.utils.data.DataLoader(
    train_ds,
    batch_size = batch_size,
    drop_last = True,
    shuffle = True,
    num_workers = 8
)
# create validation dataloader
val_dl = torch.utils.data.DataLoader(
    val_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 8
)

In [9]:
train_ds.input_cols, train_ds.output_cols

(['CRIM',
  'ZN',
  'INDUS',
  'CHAS',
  'NOX',
  'RM',
  'AGE',
  'DIS',
  'RAD',
  'TAX',
  'PTRATIO',
  'B',
  'LSTAT'],
 ['MEDV'])

# Network definition

In [10]:
# TODO: define a network composed of linear layers interleaved by ReLUs. Note: last layer must be a linear layer.

# class Net(nn.Module):
class Net(nn.Module):
  def __init__(self):
    super(Net, self).__init__()
    # define the first layer
    self.layer1 = nn.Linear(13, 128)
    self.layer2 = nn.ReLU()
    self.layer3 = nn.Linear(128, 64)
    self.layer4 = nn.ReLU()
    self.layer5 = nn.Linear(64, 1)  # the last layer must be linear

  def forward(self, x):
    # just apply the layer
    x = self.layer1(x)
    x = self.layer2(x)
    x = self.layer3(x)
    x = self.layer4(x)
    x = self.layer5(x)
    return x


In [11]:
# let's test the network
net = Net()

# define random batch of 10 elements
inp = torch.rand(10, 13)

# forward
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

 Input shape is torch.Size([10, 13])
Output shape is torch.Size([10, 1])


In [ ]:
# let's move the network in GPU
net.to(device)

# define random batch of 10 elements
inp = torch.rand(10, 13)

# move the batch in GPU
inp = inp.to(device)

# get the output
out = net(inp)

# let's print the shape
print(' Input shape is', inp.shape)
print('Output shape is', out.shape)

# Define validation routine

In [ ]:
# create validation routine
def validate(net, dl):
    # get final score
    score = 0
    # set network in eval mode
    net.eval()
    # at the end of epoch, validate model
    for inp, gt in dl:
        # move batch to gpu
        inp = inp.to(device)
        gt = gt.to(device)
        # get output
        with torch.no_grad():
            out = net(inp)
        # compare with gt
        cur_score = F.l1_loss(out, gt)
        # append
        score += cur_score
    # at the end, average over batches
    score /= len(dl)
    # set network in training mode
    net.train()
    # return score
    return score



# Train

In [ ]:
import shutil
# %load_ext tensorboard
%reload_ext tensorboard
%tensorboard --logdir={experiment_name}

In [ ]:
net, val_dl

In [ ]:
train_dl

In [ ]:
# define optimizer
optimizer = torch.optim.Adam(params=net.parameters(), lr=learning_rate)

# define summary writer
writer = SummaryWriter(experiment_name)

# initialize iteration number
n_iter = 0

# define best validation value
best_val = None

# for each epoch
for cur_epoch in range(2500):
    # plot current epoch
    writer.add_scalar("epoch", cur_epoch, n_iter)
    # TODO: for every batch, compute output, loss, perform backward propagation and finally update weights
    for inp, gt in train_dl:
      # move batches to device
      inp = inp.to(device)
      gt = gt.to(device)

      # reset the grads
      optimizer.zero_grad()
      # perform perediction
      out = net(inp)
      # compute the loss
      loss = F.l1_loss(out, gt) # l1-MAE, l2-MSE
      # copute backward
      loss.backward() # we need to compute all the gradients
      # update weights
      optimizer.step() # we optimize NN by 1 batch
      writer.add_scalar('loss', loss.item(), n_iter)
      n_iter = n_iter+1



    # at the end, validate model
    cur_val = validate(net, val_dl)
    # plot validation
    writer.add_scalar("val", loss.item(), n_iter)
    # TODO: check if it is the best model so far
    if best_val is None or cur_val > best_val:
      best_val = cur_val
      # save the current model as best
      torch.save({
          'net': net.state_dict(),
          # Save optimzer
          'optimizer': optimizer.state_dict(),
          'epoch': cur_epoch
      }, 'best.pth')
      # save last model

      torch.save({
          'net': net.state_dict(),
          # Save optimzer
          'optimizer': optimizer.state_dict(),
          'epoch': cur_epoch
      },  'best.pth')



# Test

In [ ]:
# create test dataset
test_ds =  Dataset('BostonHousingDataset/test.csv')

# create dataloader
test_dl = torch.utils.data.DataLoader(
    test_ds,
    batch_size = batch_size,
    drop_last = False,
    shuffle = False,
    num_workers = 8
)

In [ ]:
# TODO: load best network
state = torch.load('best.pth')
net.load_state_dic(state['net'])

<All keys matched successfully>

In [ ]:
test_value = validate(net, test_dl).item()

In [ ]:
print(f'The model scored a MAE of {test_value:0.04f} over the testset.')

The model scored a MAE of 0.3966 over the testset.
